# 05 · AI Discourse on Hacker News

How did the HN community's relationship with AI evolve? From sceptical blog posts about "neural networks" in 2012, through the deep learning excitement of 2016, to the ChatGPT earthquake of November 2022 — quantified.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src.loader import db
from src.nlp import keyword_hits, sentiment_score
from src.viz import set_style, save

set_style()
con = db()

In [2]:
stories = con.execute("""
    SELECT
        title,
        score,
        comment_count,
        posted_at,
        YEAR(posted_at) AS year,
        DATE_TRUNC('month', posted_at) AS month
    FROM stories
    WHERE YEAR(posted_at) BETWEEN 2012 AND 2024
      AND score >= 1
""").df()

print(f'{len(stories):,} stories')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

4,377,057 stories


## Monthly AI story volume — the ChatGPT inflection

In [3]:
AI_TERMS = {
    # Research / paradigms
    'Machine Learning': [r'machine learning', r'\bml model'],
    'Deep Learning':    ['deep learning', 'neural network'],
    'Transformer':      ['transformer model', 'attention mechanism'],
    # Models & products
    'GPT / OpenAI':     [r'\bgpt\b', 'gpt-4', 'gpt-3', 'gpt-2', 'openai'],
    'ChatGPT':          ['chatgpt'],
    'LLM':              [r'\bllm\b', 'large language model'],
    'Claude / Anthropic': ['anthropic', r'\bclaude\b ai', 'claude 3', 'claude 2'],
    'Gemini / Google':  ['gemini', 'google bard', r'\bbard\b'],
    'Copilot':          ['github copilot', r'\bcopilot\b'],
    'Stable Diffusion': ['stable diffusion', 'midjourney', r'dall[·-]e', 'imagen'],
    'Hugging Face':     ['hugging face', 'huggingface'],
    # Broader terms
    'Generative AI':    ['generative ai', 'gen ai', 'genai', 'foundation model',
                         'diffusion model', 'text-to-image', 'text to image'],
    'AI (general)':     [r'artificial intelligence'],
}

for label, patterns in AI_TERMS.items():
    stories[label] = stories['title'].apply(lambda t: keyword_hits(str(t), patterns))

stories['is_ai'] = stories[list(AI_TERMS.keys())].any(axis=1)

ai_count = stories['is_ai'].sum()
print(f'Stories matching any AI term: {ai_count:,} of {len(stories):,} ({ai_count/len(stories)*100:.1f}%)')


Stories matching any AI term: 68,299 of 4,377,057 (1.6%)


## Sentiment of AI titles over time

In [4]:
ai_stories = stories[stories[list(AI_TERMS.keys())].any(axis=1)].copy()
ai_stories['sentiment'] = ai_stories['title'].apply(sentiment_score)

sentiment_by_year = ai_stories.groupby('year')['sentiment'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#e8604c' if v > 0 else '#4c6de8' for v in sentiment_by_year.values]
ax.bar(sentiment_by_year.index, sentiment_by_year.values, color=colors, edgecolor='white')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_title('Average sentiment of AI-related HN story titles (positive = optimistic)', fontsize=13, fontweight='bold')
ax.set_ylabel('VADER compound sentiment')
plt.tight_layout()
save(fig, '../data/fig_ai_sentiment.png')
plt.show()

## Score premium: do AI stories get more upvotes?

In [5]:
stories['is_ai'] = stories[list(AI_TERMS.keys())].any(axis=1)

VIRAL_THRESHOLD = 100

viral = stories[stories['score'] >= VIRAL_THRESHOLD].copy()
viral_total = viral.groupby('year').size().rename('total_viral')
viral_ai = viral[viral['is_ai']].groupby('year').size().rename('ai_viral')

viral_share = (viral_ai / viral_total * 100).fillna(0).reset_index()
viral_share.columns = ['year', 'ai_pct_of_viral']
viral_share = viral_share[viral_share['year'].between(2015, 2024)]

print(f'AI stories as % of all stories scoring > {VIRAL_THRESHOLD} (2015–2024):')
print(viral_share.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: absolute count of viral AI stories per year
ax = axes[0]
ax.bar(viral_share['year'], viral_ai.reindex(viral_share['year']).values, color='#e8604c', edgecolor='white')
ax.set_title(f'Viral AI stories per year\n(score > {VIRAL_THRESHOLD})', fontsize=12, fontweight='bold')
ax.set_ylabel('Story count')

# Right: AI share of all viral content
ax = axes[1]
ax.plot(viral_share['year'], viral_share['ai_pct_of_viral'], linewidth=2.5, color='#e8604c', marker='o', markersize=6)
ax.fill_between(viral_share['year'], viral_share['ai_pct_of_viral'], alpha=0.15, color='#e8604c')
ax.axvline(x=2022.9, color='black', linestyle='--', alpha=0.5, linewidth=1)
ax.set_title(f'AI content as % of all viral stories\n(score > {VIRAL_THRESHOLD})', fontsize=12, fontweight='bold')
ax.set_ylabel('% of viral stories')

plt.tight_layout()
save(fig, '../data/fig_ai_score_premium.png')
plt.show()


AI stories as % of all stories scoring > 100 (2015–2024):
 year  ai_pct_of_viral
 2015         0.950145
 2016         1.943594
 2017         1.721196
 2018         1.526773
 2019         1.217079
 2020         0.925046
 2021         0.902201
 2022         1.608611
 2023         5.341063
 2024         3.867757
